In [1]:
ENV_NAME = "BipedalWalker-v3"
ENV_NUM = 16
LR = 2.5e-4

EPOCH = 1000
EPOCH_NUM_STEP = 100_000
COLLECTION_STEP_NUM_ENV_STEPS = 2048
K_EPOCH = 10

GAMMA = .99
GAE_LAMBDA = .9
EPS_CLIP = .2
MAX_GRAD_NORM = .5
VF_COEF = .5
ENTROPY_COEF = 0.001

BUFFER_SIZE: int = 2048
BATCH_SIZE: int = 64

LOG_DIR="./logs"
CHECKPOINTS_DIR = "./data/checkpoints/"

HIDDEN_SIZES = [128, 128]
IS_HARDCORE = True

print(f"{EPOCH * EPOCH_NUM_STEP:_}") 

100_000_000


In [2]:
import torch

device = "cpu"
# device = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"
device

'cpu'

In [3]:
import gymnasium as gym

In [4]:
def make_env():
    return gym.make(ENV_NAME, hardcore=IS_HARDCORE)

In [5]:
from tianshou.utils.space_info import SpaceInfo

In [6]:
env = make_env()
space_info = SpaceInfo.from_env(env)

state_shape = space_info.observation_info.obs_shape
action_shape = space_info.action_info.action_shape
max_action = space_info.action_info.max_action

space_info, state_shape

<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


(SpaceInfo(action_info=ActionSpaceInfo(action_shape=(4,), min_action=-1.0, max_action=1.0), observation_info=ObservationSpaceInfo(obs_shape=(24,))),
 (24,))

In [7]:
from tianshou.env import DummyVectorEnv, VectorEnvNormObs

In [8]:
# training envs with adjustable rms
training_envs = VectorEnvNormObs(DummyVectorEnv(
    [make_env for _ in range(ENV_NUM)]
))

# testing envs with rms copied from training
test_envs = VectorEnvNormObs(
    DummyVectorEnv([make_env for _ in range(ENV_NUM)]),
    update_obs_rms=False,
)

# copy rms. and at the end
test_envs.set_obs_rms(training_envs.get_obs_rms())

In [9]:
from tianshou.utils.net.common import Net
from torch import nn

In [10]:
net_actor = Net(
    state_shape=state_shape,
    hidden_sizes=HIDDEN_SIZES,
    activation=nn.Tanh,
)
net_critic = Net(
    state_shape=state_shape,
    hidden_sizes=HIDDEN_SIZES,
    activation=nn.Tanh,
)

In [11]:
from tianshou.utils.net.common import ActorCritic
from tianshou.utils.net.continuous import ContinuousActorProbabilistic, ContinuousCritic

In [12]:
# from docs: Simple actor network that outputs mu and sigma 
# to be used as input for a dist_fn (typically, a Gaussian).
# 
# Used primarily in SAC, PPO and variants thereof.

actor = ContinuousActorProbabilistic(
    preprocess_net=net_actor,
    action_shape=action_shape,
    unbounded=True,
).to(device)

# from docs: Simple critic network.
# It will create an actor operated in continuous 
# action space with structure of preprocess_net —> 1(q value).

critic = ContinuousCritic(preprocess_net=net_critic).to(device)

In [13]:
import numpy as np

In [14]:
# from docs: An actor-critic network for parsing parameters.
# here: just for orthogonal init!!!
actor_critic = ActorCritic(actor, critic)

nn.init.constant_(actor.sigma_param, -0.5)
for m in actor_critic.modules():
    if isinstance(m, nn.Linear):
        # orthogonal initialization
        nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
        nn.init.zeros_(m.bias)

for m in actor.mu.modules():
    if isinstance(m, nn.Linear):
        nn.init.zeros_(m.bias)
        m.weight.data.copy_(0.01 * m.weight.data)

In [15]:
import torch
from torch.distributions import Distribution, Independent, Normal

In [16]:
# this is "idiomatic" way of dist.log_prob(action).sum(-1)
# ml hackers...
def dist(loc_scale: tuple[torch.Tensor, torch.Tensor]) -> Distribution:
    loc, scale = loc_scale
    return Independent(Normal(loc, scale), 1)

In [17]:
from tianshou.algorithm.optim import AdamOptimizerFactory, LRSchedulerFactoryLinear

In [18]:
optim = AdamOptimizerFactory(lr=LR)
optim.with_lr_scheduler_factory(
    LRSchedulerFactoryLinear(
        max_epochs=EPOCH,
        epoch_num_steps=EPOCH_NUM_STEP,
        collection_step_num_env_steps=COLLECTION_STEP_NUM_ENV_STEPS,
    )
)

AdamOptimizerFactory[id=4776402896, lr_scheduler_factory=LRSchedulerFactoryLinear[num_epochs=1000, epoch_num_steps=100000, collection_step_num_env_steps=2048], lr=0.00025, weight_decay=0, eps=1e-08, betas=(0.9, 0.999)]

In [19]:
from tianshou.algorithm.modelfree.reinforce import ProbabilisticActorPolicy

In [20]:
policy = ProbabilisticActorPolicy(
    actor=actor,
    dist_fn=dist,
    action_scaling=True,
    action_bound_method="clip", # torch.clamp(-1, 1)
    action_space=env.action_space,
)

/Users/asd/Documents/dev/everything-i-reach-for/.venv/lib/python3.13/site-packages/tianshou/algorithm/modelfree/reinforce.py:152: UserWarning: action_scaling and action_bound_method are only intended to deal with unbounded model action space, but found actor model bound action space with max_action=1.0. Consider using unbounded=True option of the actor model, or set action_scaling to False and action_bound_method to None.
  warnings.warn(


In [21]:
from tianshou.algorithm import PPO

In [22]:
algorithm: PPO = PPO(
    policy=policy,
    critic=critic,
    optim=optim,
    gamma=GAMMA,
    gae_lambda=GAE_LAMBDA,
    max_grad_norm=MAX_GRAD_NORM,
    vf_coef=VF_COEF,
    ent_coef=ENTROPY_COEF,
    return_scaling=False,
    eps_clip=EPS_CLIP,
    value_clip=True,
    dual_clip=None,
    advantage_normalization=True,
    recompute_advantage=True,
)

In [23]:
from tianshou.data import Collector, CollectStats, VectorReplayBuffer

In [24]:
buffer = VectorReplayBuffer(BUFFER_SIZE, len(training_envs))
training_collector = Collector[CollectStats](
    algorithm, training_envs, buffer, exploration_noise=True
)
test_collector = Collector[CollectStats](algorithm, test_envs)

In [25]:
import datetime
import os

from tianshou.utils import TensorboardLogger
from torch.utils.tensorboard import SummaryWriter

In [26]:
now = datetime.datetime.now().strftime("%y%m%d-%H%M%S")
log_path = os.path.join(LOG_DIR, ENV_NAME, f"ppo_tianshou_{now}")
writer = SummaryWriter(log_path)
logger = TensorboardLogger(writer)

In [27]:
from tianshou.algorithm.algorithm_base import Algorithm

In [28]:
CHECKPOINT_FILE = os.path.join(CHECKPOINTS_DIR, f"policy_{ENV_NAME}.pth")

def save_best_fn(policy: Algorithm) -> None:
    state = {"model": policy.state_dict(), "obs_rms": training_envs.get_obs_rms()}
    torch.save(state, CHECKPOINT_FILE)

def load_checkpoint(file) -> None:
    state = torch.load(file, map_location="cpu", weights_only=False)
    filtered_state = {
        k[len("policy."):]: v
        for k, v in  state["model"].items()
        if k.startswith("policy.")
    }
    return filtered_state, state["obs_rms"]

class ONNXActor(torch.nn.Module):
    def __init__(self, actor, obs_rms, max_action=1.0):
        super().__init__()
        self.actor = actor
        self.register_buffer("mean", torch.as_tensor(obs_rms.mean, dtype=torch.float32))
        self.register_buffer("var", torch.as_tensor(obs_rms.var, dtype=torch.float32))
        self.eps = float(obs_rms.eps)
        self.clip_max = float(obs_rms.clip_max)
        self.max_action = max_action

    def forward(self, obs):
        obs = (obs - self.mean) / torch.sqrt(self.var + self.eps)
        obs = torch.clamp(obs, -self.clip_max, self.clip_max)
        (mu, _), _ = self.actor(obs)
        return torch.clip(mu, -1, 1) * self.max_action

def export_onnx(file, policy):
    import copy
    policy_copy = copy.deepcopy(policy)

    model_state, obs_rms = load_checkpoint(file)
    policy_copy.load_state_dict(model_state)
    model = ONNXActor(policy_copy.actor, obs_rms).eval()

    torch.onnx.export(
        model,
        torch.randn(1, state_shape[0]),
        "./data/latest.onnx",
        input_names=["obs"], output_names=["action"],
        dynamic_axes={"obs": {0: "batch"}, "action": {0: "batch"}},
        external_data=False,
    )

In [29]:
from tianshou.trainer import OnPolicyTrainerParams

In [30]:
%%capture
result = algorithm.run_training(
    OnPolicyTrainerParams(
        training_collector=training_collector,
        test_collector=test_collector,
        max_epochs=EPOCH,
        epoch_num_steps=EPOCH_NUM_STEP,
        update_step_num_repetitions=K_EPOCH,
        test_step_num_episodes=ENV_NUM,
        batch_size=BATCH_SIZE,
        collection_step_num_env_steps=COLLECTION_STEP_NUM_ENV_STEPS,
        save_best_fn=save_best_fn,
        logger=logger,
        test_in_training=False,
    )
)

KeyboardInterrupt: 

In [31]:
test_collector.reset()
s = test_collector.collect(n_episode=ENV_NUM, render=False)
print(f"reward: {s.returns_stat.mean:.1f} ± {s.returns_stat.std:.1f}")
print(f"length: {s.lens_stat.mean:.1f}")
print(f"episodes: {s.n_collected_episodes}")

reward: 1.9 ± 0.4
length: 1600.0
episodes: 16


In [32]:
%%capture
export_onnx(CHECKPOINT_FILE, policy)